# Retail Bank Analysis — Exploratory Data Analysis (EDA)

**Project:** Retail Banking Transaction Analysis  
**Role:** Data Analyst / BI Analyst

## Objective
Explore and understand the retail banking data before SQL and Power BI reporting. The analysis covers:
1. Customer Profile & Segmentation
2. Account Usage & Branch Activity
3. Transaction Patterns
4. Loan Performance & Repayment Behaviour
5. Card Usage & Product Engagement

This notebook uses the raw CSV files supplied for the project. Cleaning decisions are kept explicit so they can be reproduced later in Power Query / Power BI.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# Load the project datasets
customers = pd.read_csv("../01_Dataset/Raw/customers.csv")
accounts = pd.read_csv("../01_Dataset/Raw/accounts.csv")
branches = pd.read_csv("../01_Dataset/Raw/branches.csv")
cards = pd.read_csv("../01_Dataset/Raw/cards.csv")
loans = pd.read_csv("../01_Dataset/Raw/loans.csv")
loan_payments = pd.read_csv("../01_Dataset/Raw/loan_payments.csv")
transactions = pd.read_csv("../01_Dataset/Raw/transactions.csv")
date_table = pd.read_csv("../01_Dataset/Raw/date_table.csv")

tables = {
    "customers": customers,
    "accounts": accounts,
    "branches": branches,
    "cards": cards,
    "loans": loans,
    "loan_payments": loan_payments,
    "transactions": transactions,
    "date_table": date_table
}

for name, df in tables.items():
    print(f"{name:16} {df.shape}")


## 1. Data Understanding

The supplied project documentation gives the raw table sizes as:
- customers: 500 × 13
- accounts: 700 × 10
- branches: 40 × 7
- cards: 600 × 10
- loans: 300 × 13
- loan_payments: 2,000 × 11
- transactions: 5,000 × 10
- date_table: 6,940 rows

The purpose here is to validate those shapes against the uploaded files and inspect columns/data types.


In [ ]:
# Column and data-type overview
for name, df in tables.items():
    print("\n" + "="*80)
    print(name.upper())
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing": df.isna().sum().values,
        "unique": [df[c].nunique(dropna=True) for c in df.columns]
    }))


In [ ]:
# First five rows of each table
for name, df in tables.items():
    print(f"\n{name.upper()}")
    display(df.head())


## 2. Data Quality Checks

The project brief says the dataset contains light quality issues, especially inconsistent casing / leading and trailing spaces in categorical fields, and a small number of missing customer segments. The brief also says core outcome fields are already clean.

The checks below identify the issues without silently changing the raw data.


In [ ]:
# Missing values
missing_summary = []
for name, df in tables.items():
    for col in df.columns:
        n = df[col].isna().sum()
        if n > 0:
            missing_summary.append([name, col, int(n), round(n/len(df)*100, 2)])

missing_df = pd.DataFrame(
    missing_summary,
    columns=["table", "column", "missing_count", "missing_pct"]
).sort_values(["table", "missing_count"], ascending=[True, False])

display(missing_df)


In [ ]:
# Duplicate rows and duplicate key candidates
for name, df in tables.items():
    print(f"{name}: duplicate rows = {df.duplicated().sum()}")

key_candidates = {
    "customers": "customer_id",
    "accounts": "account_id",
    "branches": "branch_id",
    "cards": "card_id",
    "loans": "loan_id",
    "loan_payments": "payment_id",
    "transactions": "transaction_id",
    "date_table": "Date"
}

for name, key in key_candidates.items():
    if key in tables[name].columns:
        dup = tables[name][key].duplicated().sum()
        print(f"{name}.{key}: duplicate keys = {dup}")


In [ ]:
# Leading/trailing whitespace in important categorical columns
categorical_candidates = {
    "customers": ["gender", "kyc_status", "segment"],
    "accounts": ["account_type", "status"],
    "cards": ["card_type", "network", "is_active"],
    "loans": ["loan_type", "purpose", "loan_status"],
    "loan_payments": ["payment_method", "status"],
    "transactions": ["transaction_type", "channel", "description", "status"]
}

for table, cols in categorical_candidates.items():
    print(f"\n{table}")
    for col in cols:
        if col in tables[table].columns:
            n = (tables[table][col].astype("string") != tables[table][col].astype("string").str.strip()).sum()
            print(f"{col}: {n} values with leading/trailing whitespace")


## 3. Non-destructive EDA Cleaning View

For EDA, create analysis copies. The original raw CSVs remain unchanged.

The project brief recommends Trim/Clean and consistent capitalization for descriptive categorical fields. For missing customer segments, the brief gives **Unspecified** as an example rather than silently dropping rows.


In [ ]:
# Create analysis copies
eda = {name: df.copy() for name, df in tables.items()}

# Strip whitespace from object/string columns
for name, df in eda.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

# Explicitly preserve missing customer segment as a category for analysis
if "segment" in eda["customers"].columns:
    eda["customers"]["segment"] = eda["customers"]["segment"].fillna("Unspecified")

# Parse likely date columns
for name, df in eda.items():
    for col in df.columns:
        if "date" in col.lower() or col.lower() == "date":
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce")
            except Exception:
                pass

# Use the cleaned analysis copies
customers, accounts, branches, cards, loans, loan_payments, transactions, date_table = (
    eda["customers"], eda["accounts"], eda["branches"], eda["cards"],
    eda["loans"], eda["loan_payments"], eda["transactions"], eda["date_table"]
)


# 4. Customer Profile & Segmentation

Business objective: understand the customer base and differences between customer groups.

We examine:
- Segments
- Demographics
- Geography
- Income
- Credit score
- KYC
- Activity
- Customer tenure


In [ ]:
# Customer KPIs
customer_kpis = pd.Series({
    "Total Customers": customers["customer_id"].nunique(),
    "Active Customers": (customers["is_active"].astype(str).str.lower() == "yes").sum(),
    "Average Annual Income": customers["annual_income"].mean(),
    "Average Credit Score": customers["credit_score"].mean()
})
display(customer_kpis.to_frame("value"))


In [ ]:
# Segment profile
segment_profile = customers.groupby("segment", dropna=False).agg(
    customers=("customer_id", "nunique"),
    avg_income=("annual_income", "mean"),
    avg_credit_score=("credit_score", "mean")
).sort_values("customers", ascending=False)
display(segment_profile)


In [ ]:
# Customers by state
state_profile = customers.groupby("state").agg(
    customers=("customer_id", "nunique"),
    active_customers=("is_active", lambda s: (s.astype(str).str.lower()=="yes").sum())
).sort_values("customers", ascending=False)
display(state_profile)


In [ ]:
# KYC distribution
display(customers["kyc_status"].value_counts(dropna=False).to_frame("customers"))


In [ ]:
# Customer tenure using latest transaction date as project "today"
latest_transaction_date = transactions["transaction_date"].max()
customers["tenure_years"] = (
    (latest_transaction_date - customers["customer_since"]).dt.days / 365.25
)

customers["tenure_band"] = pd.cut(
    customers["tenure_years"],
    bins=[-np.inf, 3, 6, 10, np.inf],
    labels=["0-2 Years", "3-5 Years", "6-9 Years", "10+ Years"],
    right=False
)

display(customers["tenure_band"].value_counts().sort_index())


In [ ]:
# Visual: customers by segment
customers["segment"].value_counts(dropna=False).plot(kind="bar")
plt.title("Customers by Segment")
plt.xlabel("Segment")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.show()


# 5. Account Usage & Branch Activity

Business objective: understand account usage and how activity differs across account types and branches.


In [ ]:
account_kpis = pd.Series({
    "Total Accounts": accounts["account_id"].nunique(),
    "Total Current Balance": accounts["current_balance"].sum(),
    "Average Account Balance": accounts["current_balance"].mean(),
    "Active Accounts": (accounts["status"].astype(str).str.lower() == "active").sum()
})
display(account_kpis.to_frame("value"))


In [ ]:
# Account type analysis
account_type = accounts.groupby("account_type").agg(
    accounts=("account_id", "nunique"),
    total_balance=("current_balance", "sum"),
    avg_balance=("current_balance", "mean"),
    avg_interest_rate=("interest_rate", "mean")
).sort_values("total_balance", ascending=False)

display(account_type)


In [ ]:
# Branch analysis
branch_activity = (
    accounts.merge(
        branches[["branch_id", "branch_name", "region"]],
        on="branch_id",
        how="left"
    )
    .groupby(["branch_id", "branch_name", "region"])
    .agg(
        accounts=("account_id", "nunique"),
        total_balance=("current_balance", "sum"),
        avg_balance=("current_balance", "mean")
    )
    .sort_values("total_balance", ascending=False)
)
display(branch_activity.head(15))


In [ ]:
# Visual: account type balances
account_type["total_balance"].plot(kind="bar")
plt.title("Total Account Balance by Account Type")
plt.xlabel("Account Type")
plt.ylabel("Balance")
plt.xticks(rotation=30)
plt.show()


# 6. Transaction Pattern Analysis

Business objective: understand how customers use accounts and how money moves through the banking system.


In [ ]:
transaction_kpis = pd.Series({
    "Total Transactions": transactions["transaction_id"].nunique(),
    "Completed Transactions": (transactions["status"].astype(str).str.lower() == "completed").sum(),
    "Completed Transaction Amount": transactions.loc[
        transactions["status"].astype(str).str.lower() == "completed", "amount"
    ].sum(),
    "Average Transaction Amount": transactions["amount"].mean()
})
display(transaction_kpis.to_frame("value"))


In [ ]:
# Transaction types
transaction_type = transactions.groupby("transaction_type").agg(
    transactions=("transaction_id", "nunique"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean")
).sort_values("total_amount", ascending=False)
display(transaction_type)


In [ ]:
# Transaction channels
channel_profile = transactions.groupby("channel").agg(
    transactions=("transaction_id", "nunique"),
    total_amount=("amount", "sum")
).sort_values("transactions", ascending=False)
display(channel_profile)


In [ ]:
# Transaction status
display(
    transactions.groupby("status").agg(
        transactions=("transaction_id","nunique"),
        total_amount=("amount","sum")
    ).sort_values("transactions", ascending=False)
)


In [ ]:
# Monthly trend
transactions["year_month"] = transactions["transaction_date"].dt.to_period("M").astype(str)
monthly_transactions = transactions.groupby("year_month").agg(
    transactions=("transaction_id", "nunique"),
    completed_amount=("amount", lambda s: s[transactions.loc[s.index, "status"].astype(str).str.lower()=="completed"].sum())
)
display(monthly_transactions.head(12))


In [ ]:
# Visual: monthly transaction count
monthly_transactions["transactions"].plot(kind="line", marker="o")
plt.title("Monthly Transaction Volume")
plt.xlabel("Month")
plt.ylabel("Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 7. Loan Performance & Repayment Behaviour

Business objective: understand loan performance and whether customers are repaying as expected.


In [ ]:
loan_kpis = pd.Series({
    "Total Loans": loans["loan_id"].nunique(),
    "Total Principal": loans["principal_amount"].sum(),
    "Total Outstanding": loans["outstanding_balance"].sum(),
    "Average Loan Amount": loans["principal_amount"].mean()
})
display(loan_kpis.to_frame("value"))


In [ ]:
# Loan type portfolio
loan_type = loans.groupby("loan_type").agg(
    loans=("loan_id", "nunique"),
    principal=("principal_amount", "sum"),
    outstanding=("outstanding_balance", "sum"),
    avg_rate=("interest_rate", "mean")
).sort_values("outstanding", ascending=False)
display(loan_type)


In [ ]:
# Loan status
display(
    loans.groupby("loan_status").agg(
        loans=("loan_id","nunique"),
        outstanding=("outstanding_balance","sum")
    ).sort_values("outstanding", ascending=False)
)


In [ ]:
# Repayment behaviour
payment_profile = loan_payments.groupby("status").agg(
    payments=("payment_id","nunique"),
    scheduled_amount=("scheduled_amount","sum"),
    paid_amount=("paid_amount","sum"),
    avg_days_late=("days_late","mean")
).sort_values("payments", ascending=False)
display(payment_profile)


In [ ]:
# Late payment rate by loan type
loan_payment_detail = loan_payments.merge(
    loans[["loan_id","loan_type"]],
    on="loan_id",
    how="left"
)
late_by_type = loan_payment_detail.groupby("loan_type").agg(
    payments=("payment_id","nunique"),
    late_payments=("days_late", lambda s: (s > 0).sum()),
    avg_days_late=("days_late","mean"),
    penalty=("penalty","sum")
)
late_by_type["late_rate_pct"] = late_by_type["late_payments"] / late_by_type["payments"] * 100
display(late_by_type.sort_values("late_rate_pct", ascending=False))


In [ ]:
# Visual: outstanding balance by loan type
loan_type["outstanding"].plot(kind="bar")
plt.title("Outstanding Loan Balance by Loan Type")
plt.xlabel("Loan Type")
plt.ylabel("Outstanding Balance")
plt.xticks(rotation=30)
plt.show()


# 8. Card Usage & Product Engagement

Business objective: understand card usage and how card products relate to the broader banking relationship.


In [ ]:
card_kpis = pd.Series({
    "Total Cards": cards["card_id"].nunique(),
    "Active Cards": (cards["is_active"].astype(str).str.lower() == "yes").sum(),
    "Total Credit Limit": cards["credit_limit"].sum(),
    "Total Card Outstanding": cards["outstanding_balance"].sum(),
    "Total Reward Points": cards["reward_points"].sum()
})
display(card_kpis.to_frame("value"))


In [ ]:
# Card type profile
card_type = cards.groupby("card_type").agg(
    cards=("card_id","nunique"),
    active_cards=("is_active", lambda s: (s.astype(str).str.lower()=="yes").sum()),
    avg_credit_limit=("credit_limit","mean"),
    outstanding=("outstanding_balance","sum"),
    reward_points=("reward_points","sum")
).sort_values("cards", ascending=False)
display(card_type)


In [ ]:
# Card network profile
network_profile = cards.groupby("network").agg(
    cards=("card_id","nunique"),
    outstanding=("outstanding_balance","sum"),
    reward_points=("reward_points","sum")
).sort_values("cards", ascending=False)
display(network_profile)


In [ ]:
# Accounts without a card
accounts_with_cards = cards["account_id"].nunique()
accounts_without_cards = accounts["account_id"].nunique() - accounts_with_cards
print("Accounts:", accounts["account_id"].nunique())
print("Accounts with cards:", accounts_with_cards)
print("Accounts without cards:", accounts_without_cards)
print("Percentage without cards:", round(accounts_without_cards/accounts["account_id"].nunique()*100, 2), "%")


# 9. Cross-Product Engagement

Connect customers, accounts, cards, and loans to understand the breadth of the banking relationship.

This section is descriptive: it identifies product ownership/usage patterns and does not claim causation.


In [ ]:
customer_products = (
    customers[["customer_id","segment"]]
    .merge(accounts.groupby("customer_id")["account_id"].nunique().rename("accounts"), on="customer_id", how="left")
    .merge(
        accounts[["account_id","customer_id"]]
        .merge(cards[["card_id","account_id"]], on="account_id", how="inner")
        .groupby("customer_id")["card_id"].nunique()
        .rename("cards"),
        on="customer_id", how="left"
    )
    .merge(loans.groupby("customer_id")["loan_id"].nunique().rename("loans"), on="customer_id", how="left")
    .fillna(0)
)

customer_products["product_count"] = (
    (customer_products["accounts"] > 0).astype(int)
    + (customer_products["cards"] > 0).astype(int)
    + (customer_products["loans"] > 0).astype(int)
)

display(customer_products.groupby("segment").agg(
    customers=("customer_id","nunique"),
    avg_accounts=("accounts","mean"),
    avg_cards=("cards","mean"),
    avg_loans=("loans","mean"),
    avg_product_count=("product_count","mean")
).sort_values("avg_product_count", ascending=False))


# 10. Key EDA Findings

Populate this section after reviewing the outputs above. Use the structure:

- **Observation:** What does the data show?
- **Evidence:** Which metric/table supports it?
- **Business implication:** Why might it matter?
- **Caution:** Avoid claiming causation when the EDA only shows association.

These findings will feed the SQL insights and Power BI report.


In [ ]:
# Compact KPI summary for use in the final report
summary = pd.DataFrame({
    "Metric": [
        "Customers",
        "Accounts",
        "Branches",
        "Cards",
        "Loans",
        "Loan Payments",
        "Transactions",
        "Total Outstanding Loan Balance"
    ],
    "Value": [
        customers["customer_id"].nunique(),
        accounts["account_id"].nunique(),
        branches["branch_id"].nunique(),
        cards["card_id"].nunique(),
        loans["loan_id"].nunique(),
        loan_payments["payment_id"].nunique(),
        transactions["transaction_id"].nunique(),
        loans["outstanding_balance"].sum()
    ]
})
display(summary)
